# Dataset discovery and text preparation

Historical research source, original zero-based cells **0?10**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 0

Position 1 in the original notebook; original code comments may use different numbering.


In [ ]:

import torch
import platform
import shutil

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print("\nGPU:", torch.cuda.get_device_name(0))
    print("GPU VRAM:", round(gpu.total_memory / 1024**3, 2), "GB")
    print("CUDA version:", torch.version.cuda)
else:
    print("\nWARNING: GPU is not enabled.")

disk = shutil.disk_usage("/content")

print("\nColab storage:")
print("Total:", round(disk.total / 1024**3, 2), "GB")
print("Free:", round(disk.free / 1024**3, 2), "GB")


## Original source cell 1

Position 2 in the original notebook; original code comments may use different numbering.


In [ ]:

%pip -q install "transformers==4.57.6" "datasets[audio]==4.8.5" "accelerate==1.15.0" "torchcodec==0.11.0" "jiwer==4.0.0" "soundfile>=0.13.1,<1" "librosa>=0.11,<1"

import importlib.metadata as metadata
import shutil

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "torchcodec",
    "jiwer",
    "soundfile",
    "librosa",
]

print("Installed libraries:")
for package in packages:
    print(f"{package}: {metadata.version(package)}")

from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)
from datasets import load_dataset, Audio
from torchcodec.decoders import AudioDecoder

print("\nASR imports: successful")
print("FFmpeg available:", shutil.which("ffmpeg") is not None)


## Original source cell 2

Position 3 in the original notebook; original code comments may use different numbering.


In [ ]:

from datasets import load_dataset_builder

DATASET_NAME = "google/WaxalNLP"
DATASET_CONFIG = "amh_asr"

print("Inspecting dataset metadata...\n")

builder = load_dataset_builder(
    DATASET_NAME,
    DATASET_CONFIG
)

info = builder.info

print("Dataset:", DATASET_NAME)
print("Configuration:", builder.config.name)

print("\nFeatures:")
print(info.features)

print("\nAvailable splits:")

if info.splits:
    for name, split in info.splits.items():
        print(f"\n{name}:")
        print("  Recordings:", split.num_examples)
        print(
            "  Storage size:",
            round(split.num_bytes / 1024**3, 2)
            if split.num_bytes else "Unknown",
            "GB"
        )
else:
    print("Split counts unavailable in builder metadata.")

print("\nLicense:", info.license or "See dataset card")


## Original source cell 3

Position 4 in the original notebook; original code comments may use different numbering.


In [ ]:

import requests

DATASET_NAME = "google/WaxalNLP"
DATASET_CONFIG = "amh_asr"

url = "https://datasets-server.huggingface.co/size"

response = requests.get(
    url,
    params={"dataset": DATASET_NAME},
    timeout=60
)
response.raise_for_status()

data = response.json()
size_info = data.get("size", {})

# Find our specific language configuration.
configs = [
    c for c in size_info.get("configs", [])
    if c.get("config") == DATASET_CONFIG
]

splits = [
    s for s in size_info.get("splits", [])
    if s.get("config") == DATASET_CONFIG
]

print("Dataset:", DATASET_NAME)
print("Configuration:", DATASET_CONFIG)

print("\nConfiguration summary:")
for config in configs:
    print("Total recordings:", config.get("num_rows"))
    print(
        "Storage:",
        round(
            (config.get("num_bytes_original_files") or 0)
            / 1024**3, 2
        ),
        "GB"
    )

print("\nSplit details:")
for split in splits:
    print(
        f"{split.get('split', 'unknown'):12} "
        f"recordings={split.get('num_rows', 'unknown')}, "
        f"size={((split.get('num_bytes_original_files')))} "
        f"or {(split.get('num_bytes_parquet_files') or 0) / 1024**3:.2f} GB"
    )

print("\nPartial metadata:", data.get("partial", False))
print("Pending entries:", len(data.get("pending", [])))
print("Failed entries:", len(data.get("failed", [])))

if not splits:
    print("\nNo split counts returned by the API.")
    print("We may need to inspect Parquet metadata directly.")


## Original source cell 4

Position 5 in the original notebook; original code comments may use different numbering.


In [ ]:

import requests
import fsspec
import pyarrow.parquet as pq

# Get the Parquet files for the Amharic training split.
response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": "google/WaxalNLP"},
    timeout=60
)
response.raise_for_status()

metadata = response.json()

train_shards = [
    entry
    for entry in metadata["parquet_files"]
    if entry["config"] == "amh_asr"
    and entry["split"] == "train"
]

train_shards.sort(key=lambda x: x["filename"])

print("Training Parquet shards:", len(train_shards))
print(
    "Total shard size:",
    round(sum(s["size"] for s in train_shards) / 1024**3, 2),
    "GB"
)

assert train_shards, "No Amharic training shards found."

# Read only metadata columns from the first shard.
url = train_shards[0]["url"]

with fsspec.open(url, "rb", block_size=8 * 1024**2) as file:
    parquet = pq.ParquetFile(file)

    print("\nAvailable columns:")
    print(parquet.schema_arrow.names)

    table = parquet.read_row_group(
        0,
        columns=["id", "speaker_id", "transcription"]
    )

    rows = table.slice(0, 3).to_pylist()

print("\nFirst three examples:")
for row in rows:
    print("\nID:", row["id"])
    print("Speaker:", row["speaker_id"])
    print("Text:", row["transcription"][:200])


## Original source cell 5

Position 6 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
from pathlib import Path
from collections import Counter

import fsspec
import pyarrow.parquet as pq
from tqdm.auto import tqdm

OUTPUT_PATH = Path("/content/waxal_amh_train_metadata.jsonl")

total_rows = 0
empty_texts = 0
duplicate_ids = 0

speaker_ids = set()
seen_ids = set()
character_counts = Counter()

with OUTPUT_PATH.open("w", encoding="utf-8") as out:

    for shard in tqdm(train_shards, desc="Reading training shards"):

        with fsspec.open(
            shard["url"],
            "rb",
            block_size=8 * 1024**2
        ) as file:

            parquet = pq.ParquetFile(file)

            for batch in parquet.iter_batches(
                batch_size=512,
                columns=["id", "speaker_id", "transcription"]
            ):
                for record in batch.to_pylist():

                    record_id = record["id"]
                    speaker_id = record["speaker_id"]
                    text = record["transcription"] or ""

                    total_rows += 1

                    if not text.strip():
                        empty_texts += 1

                    if record_id in seen_ids:
                        duplicate_ids += 1

                    seen_ids.add(record_id)
                    speaker_ids.add(speaker_id)

                    character_counts.update(text)

                    out.write(
                        json.dumps(
                            {
                                "id": record_id,
                                "speaker_id": speaker_id,
                                "transcription": text
                            },
                            ensure_ascii=False
                        ) + "\n"
                    )

print("\nDATASET SUMMARY")
print("Total recordings:", total_rows)
print("Unique speakers:", len(speaker_ids))
print("Empty transcriptions:", empty_texts)
print("Duplicate IDs:", duplicate_ids)
print("Unique characters:", len(character_counts))

print(
    "Metadata file size:",
    round(OUTPUT_PATH.stat().st_size / 1024**2, 2),
    "MB"
)

print("\nTop 15 characters:")
for char, count in character_counts.most_common(15):
    print(repr(char), count)

assert total_rows == 38022, (
    f"Expected 38,022 rows, found {total_rows}"
)


## Original source cell 6

Position 7 in the original notebook; original code comments may use different numbering.


In [ ]:

import unicodedata
import json
from collections import Counter

def character_group(char):
    name = unicodedata.name(char, "UNKNOWN")
    category = unicodedata.category(char)

    if char.isspace():
        return "WHITESPACE"

    if name.startswith("ETHIOPIC"):
        return "ETHIOPIC"

    if "LATIN" in name:
        return "LATIN"

    if category.startswith("N"):
        return "NUMBER"

    if category.startswith("P"):
        return "PUNCTUATION"

    if category.startswith("S"):
        return "SYMBOL"

    if category.startswith(("C", "M")):
        return "CONTROL / MARK"

    return "OTHER"


# 1. Group characters by Unicode type.
groups = {}

for char, count in character_counts.items():
    group = character_group(char)

    if group not in groups:
        groups[group] = {
            "characters": 0,
            "occurrences": 0
        }

    groups[group]["characters"] += 1
    groups[group]["occurrences"] += count


print("CHARACTER GROUPS")
print("-" * 55)

for group, stats in sorted(groups.items()):
    print(
        f"{group:18} "
        f"unique={stats['characters']:3} "
        f"occurrences={stats['occurrences']:,}"
    )


# 2. Inspect characters outside the Ethiopic script.
print("\nNON-ETHIOPIC CHARACTERS")
print("-" * 55)

non_ethiopic = [
    (char, count)
    for char, count in character_counts.items()
    if character_group(char)
    not in ("ETHIOPIC", "WHITESPACE")
]

for char, count in sorted(
    non_ethiopic,
    key=lambda x: -x[1]
)[:60]:
    print(
        f"{repr(char):10} "
        f"{count:8,} "
        f"{unicodedata.name(char, 'UNKNOWN')}"
    )


# 3. Inspect rare characters.
rare = [
    (char, count)
    for char, count in character_counts.items()
    if count <= 5
]

print("\nRARE CHARACTERS (frequency <= 5)")
print("Total:", len(rare))

for char, count in sorted(
    rare, key=lambda x: x[1]
)[:40]:
    print(
        repr(char),
        count,
        unicodedata.name(char, "UNKNOWN")
    )


# 4. Check normalization effects.
nfc_changes = 0
nfkc_changes = 0

with open(
    OUTPUT_PATH,
    encoding="utf-8"
) as file:
    for line in file:
        text = json.loads(line)["transcription"]

        if unicodedata.normalize("NFC", text) != text:
            nfc_changes += 1

        if unicodedata.normalize("NFKC", text) != text:
            nfkc_changes += 1

print("\nUNICODE NORMALIZATION")
print("Transcripts changed by NFC:", nfc_changes)
print("Transcripts changed by NFKC:", nfkc_changes)


## Original source cell 7

Position 8 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import unicodedata
from pathlib import Path
from collections import Counter

NORMALIZED_PATH = Path(
    "/content/waxal_amh_train_normalized.jsonl"
)

def normalize_amharic_ctc(text):
    text = unicodedata.normalize("NFC", text)

    result = []

    for char in text:
        name = unicodedata.name(char, "")
        category = unicodedata.category(char)

        # Standardize whitespace.
        if char.isspace():
            result.append(" ")
            continue

        # Preserve Ethiopic letters and numerals.
        if name.startswith((
            "ETHIOPIC SYLLABLE",
            "ETHIOPIC LETTER",
            "ETHIOPIC DIGIT",
            "ETHIOPIC NUMBER"
        )):
            result.append(char)
            continue

        # Preserve Latin letters, including accented ones.
        if "LATIN" in name and category.startswith("L"):
            result.append(char.lower())
            continue

        # Preserve ASCII digits.
        if char in "0123456789":
            result.append(char)
            continue

        # Remove invisible control/format characters.
        if category.startswith("C"):
            continue

        # Treat punctuation, symbols, and other scripts
        # as word separators.
        result.append(" ")

    # Collapse consecutive spaces.
    return " ".join("".join(result).split())


normalized_counts = Counter()
changed = 0
empty = 0
examples = []

with OUTPUT_PATH.open("r", encoding="utf-8") as source, \
     NORMALIZED_PATH.open("w", encoding="utf-8") as destination:

    for line in source:
        record = json.loads(line)

        original = record["transcription"]
        normalized = normalize_amharic_ctc(original)

        if normalized != original:
            changed += 1

            if len(examples) < 5:
                examples.append((original, normalized))

        if not normalized:
            empty += 1

        normalized_counts.update(normalized)

        record["transcription"] = normalized

        destination.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


print("NORMALIZATION SUMMARY")
print("-" * 45)

print("Original unique characters:", len(character_counts))
print("Normalized unique characters:", len(normalized_counts))
print("Modified transcriptions:", changed)
print("Empty transcriptions:", empty)

print(
    "Normalized metadata size:",
    round(NORMALIZED_PATH.stat().st_size / 1024**2, 2),
    "MB"
)

print("\nCHARACTER COVERAGE")
print("Contains reserved '|':", "|" in normalized_counts)

print("\nEXAMPLE TRANSFORMATIONS")

for i, (before, after) in enumerate(examples, 1):
    print(f"\nExample {i}")
    print("BEFORE:", before[:180])
    print("AFTER: ", after[:180])

assert "|" not in normalized_counts
assert empty == 0


## Original source cell 8

Position 9 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
from pathlib import Path
from transformers import Wav2Vec2CTCTokenizer

TOKENIZER_DIR = Path("/content/amharic_ctc_tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

# Build vocabulary using the complete normalized corpus.
characters = sorted(
    char for char in normalized_counts
    if char != " "
)

vocab = {
    char: index
    for index, char in enumerate(characters)
}

# Reserve special tokens.
vocab["|"] = len(vocab)
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with VOCAB_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        vocab,
        file,
        ensure_ascii=False,
        indent=2
    )

# Explicitly disable unnecessary BOS/EOS tokens.
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
    clean_up_tokenization_spaces=False
)

# Verify vocabulary integrity.
assert len(tokenizer) == len(vocab)
assert len(set(vocab.values())) == len(vocab)
assert set(vocab.values()) == set(range(len(vocab)))

# All normalized training characters must be representable.
missing = set(normalized_counts) - {" "} - set(vocab)
assert not missing, f"Missing characters: {missing}"

# Test real training transcriptions.
with NORMALIZED_PATH.open(encoding="utf-8") as file:
    samples = [
        json.loads(next(file))["transcription"]
        for _ in range(3)
    ]

for i, text in enumerate(samples, 1):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    # Disable CTC duplicate collapsing for label round-trip.
    decoded = tokenizer.decode(
        token_ids,
        group_tokens=False,
        clean_up_tokenization_spaces=False
    )

    assert tokenizer.unk_token_id not in token_ids
    assert decoded == text, f"Round-trip failed: {i}"

    print(f"Example {i}:")
    print("  Characters:", len(text))
    print("  Tokens:", len(token_ids))
    print("  Round-trip successful:", decoded == text)

# Save tokenizer configuration alongside vocabulary.
tokenizer.save_pretrained(str(TOKENIZER_DIR))

print("\nVOCABULARY SUMMARY")
print("Total tokens:", len(tokenizer))
print("Word delimiter ID:", tokenizer.word_delimiter_token_id)
print("Unknown token ID:", tokenizer.unk_token_id)
print("CTC blank ID:", tokenizer.pad_token_id)
print("BOS token:", tokenizer.bos_token)
print("EOS token:", tokenizer.eos_token)
print("Saved to:", TOKENIZER_DIR)


## Original source cell 9

Position 10 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import unicodedata
from pathlib import Path
from collections import Counter

import fsspec
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from transformers import Wav2Vec2CTCTokenizer

# 1. Verify that the saved tokenizer reloads identically.
reloaded_tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    str(TOKENIZER_DIR)
)

assert reloaded_tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_tokenizer.pad_token_id == 320
assert reloaded_tokenizer.unk_token_id == 319

print("Tokenizer reload: PASSED")

# 2. Locate validation Parquet shards.
val_shards = sorted(
    [
        shard
        for shard in metadata["parquet_files"]
        if shard["config"] == "amh_asr"
        and shard["split"] == "validation"
    ],
    key=lambda shard: shard["filename"]
)

assert val_shards, "No validation shards found."

# 3. Scan validation text without loading audio.
VAL_METADATA_PATH = Path(
    "/content/waxal_amh_validation_normalized.jsonl"
)

oov_counts = Counter()
validation_speakers = set()

total_records = 0
total_characters = 0
oov_characters = 0
affected_records = 0
empty_records = 0

with VAL_METADATA_PATH.open("w", encoding="utf-8") as out:
    for shard in tqdm(val_shards, desc="Scanning validation"):
        with fsspec.open(
            shard["url"], "rb", block_size=8 * 1024**2
        ) as file:
            parquet = pq.ParquetFile(file)

            for batch in parquet.iter_batches(
                batch_size=512,
                columns=["id", "speaker_id", "transcription"]
            ):
                for record in batch.to_pylist():
                    text = normalize_amharic_ctc(
                        record["transcription"] or ""
                    )

                    total_records += 1
                    total_characters += len(text)
                    validation_speakers.add(record["speaker_id"])

                    if not text:
                        empty_records += 1

                    # Spaces are represented by | in the tokenizer.
                    missing = [
                        char for char in text
                        if char != " " and char not in vocab
                    ]

                    if missing:
                        affected_records += 1
                        oov_characters += len(missing)
                        oov_counts.update(missing)

                    out.write(
                        json.dumps(
                            {
                                "id": record["id"],
                                "speaker_id": record["speaker_id"],
                                "transcription": text
                            },
                            ensure_ascii=False
                        ) + "\n"
                    )

print("\nVALIDATION VOCABULARY AUDIT")
print("-" * 45)
print("Validation recordings:", total_records)
print("Empty transcriptions:", empty_records)
print("Unique validation speakers:", len(validation_speakers))
print(
    "Speakers overlapping training:",
    len(validation_speakers & speaker_ids)
)
print("Recordings containing OOV characters:", affected_records)
print("Unique unseen characters:", len(oov_counts))
print("Total OOV character occurrences:", oov_characters)
print(
    "OOV character rate:",
    f"{100 * oov_characters / max(total_characters, 1):.4f}%"
)

print("\nUNSEEN CHARACTERS (top 20)")
for char, count in oov_counts.most_common(20):
    print(
        repr(char),
        count,
        unicodedata.name(char, "UNKNOWN")
    )

assert total_records == 2912
assert empty_records == 0

print("\nSaved validation metadata:", VAL_METADATA_PATH)


## Original source cell 10

Position 11 in the original notebook; original code comments may use different numbering.


In [ ]:

from datasets import load_dataset
from torchcodec.decoders import AudioDecoder
from collections import Counter
from itertools import islice
import statistics

# Stream only the first training shard.
# Avoid loading all 21 shards into memory.
audio_stream = load_dataset(
    "parquet",
    data_files={"train": train_shards[0]["url"]},
    split="train",
    streaming=True
).decode(False)

audio_probe = []

for example in islice(audio_stream, 12):
    audio = example["audio"]

    # Parquet generally stores the encoded audio bytes.
    source = (
        audio["bytes"]
        if audio["bytes"] is not None
        else audio["path"]
    )

    decoder = AudioDecoder(source)
    info = decoder.metadata

    duration = info.duration_seconds

    audio_probe.append({
        "id": example["id"],
        "duration": duration,
        "sample_rate": info.sample_rate,
        "channels": info.num_channels,
        "codec": info.codec
    })

print("AUDIO PROBE RESULTS")
print("-" * 50)

for i, item in enumerate(audio_probe, 1):
    print(
        f"{i:02d} | "
        f"{item['duration']:.2f}s | "
        f"{item['sample_rate']} Hz | "
        f"{item['channels']} channel(s) | "
        f"{item['codec']}"
    )

durations = [x["duration"] for x in audio_probe]

print("\nSUMMARY")
print("Recordings inspected:", len(audio_probe))
print("Mean duration:", round(statistics.mean(durations), 2), "s")
print("Median duration:", round(statistics.median(durations), 2), "s")
print("Min duration:", round(min(durations), 2), "s")
print("Max duration:", round(max(durations), 2), "s")

print(
    "Sample rates:",
    dict(Counter(x["sample_rate"] for x in audio_probe))
)

print(
    "Channels:",
    dict(Counter(x["channels"] for x in audio_probe))
)

print(
    "Codecs:",
    dict(Counter(x["codec"] for x in audio_probe))
)
